In [ ]:
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import os
import re


In [ ]:
# GPU CHECK
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# LOAD DATA
def load_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            out.append(json.loads(line))
    return out

train_raw = load_jsonl("../../data/train.jsonl")
test_raw  = load_jsonl("../../data/kaggle_test.jsonl")

len(train_raw), len(test_raw)



In [ ]:
def extract_text(tweet):
    """
    Priorité :
    - extended_tweet.full_text
    - text
    - quoted_status.extended_tweet.full_text
    - quoted_status.text
    """
    if "extended_tweet" in tweet and tweet["extended_tweet"]:
        if "full_text" in tweet["extended_tweet"]:
            return tweet["extended_tweet"]["full_text"]

    if "text" in tweet and tweet["text"]:
        return tweet["text"]

    if "quoted_status" in tweet and tweet["quoted_status"]:
        q = tweet["quoted_status"]
        if "extended_tweet" in q and q["extended_tweet"] and "full_text" in q["extended_tweet"]:
            return q["extended_tweet"]["full_text"]
        if "text" in q:
            return q["text"]

    return ""


In [ ]:
def count_hashtags(entities):
    try:
        return len(entities.get("hashtags", []))
    except:
        return 0

def count_urls(entities):
    try:
        return len(entities.get("urls", []))
    except:
        return 0

def count_user_mentions(entities):
    try:
        return len(entities.get("user_mentions", []))
    except:
        return 0

def safe_get(d, key, default=0):
    return d.get(key, default) if isinstance(d, dict) else default

def extract_user_features(user):
    """
    Retourne un dict de features numériques pertinents du user.
    """
    if user is None or not isinstance(user, dict):
        return {
            "user_followers": 0,
            "user_friends": 0,
            "user_statuses": 0,
            "user_verified": 0,
            "user_listed": 0,
            "user_favourites": 0,
            "user_geo_enabled": 0,
        }

    return {
        "user_followers": safe_get(user, "followers_count", 0),
        "user_friends": safe_get(user, "friends_count", 0),
        "user_statuses": safe_get(user, "statuses_count", 0),
        "user_verified": int(safe_get(user, "verified", False)),
        "user_listed": safe_get(user, "listed_count", 0),
        "user_favourites": safe_get(user, "favourites_count", 0),
        "user_geo_enabled": int(safe_get(user, "geo_enabled", False)),
    }

def extract_quoted_user_features(quoted_status):
    if quoted_status is None or "user" not in quoted_status:
        return {
            "q_user_followers": 0,
            "q_user_friends": 0,
            "q_user_statuses": 0,
            "q_user_verified": 0,
        }
    u = quoted_status["user"]

    return {
        "q_user_followers": safe_get(u, "followers_count", 0),
        "q_user_friends": safe_get(u, "friends_count", 0),
        "q_user_statuses": safe_get(u, "statuses_count", 0),
        "q_user_verified": int(safe_get(u, "verified", False)),
    }

def extract_tweet_features(t):
    entities = t.get("entities", {})
    quoted  = t.get("quoted_status", None)

    feats = {
        "len_text": len(extract_text(t)),
        "has_extended": int("extended_tweet" in t and t["extended_tweet"] is not None),
        "n_hashtags": count_hashtags(entities),
        "n_urls": count_urls(entities),
        "n_mentions": count_user_mentions(entities),

        "favorite_count": safe_get(t, "favorite_count", 0),
        "retweet_count": safe_get(t, "retweet_count", 0),
        "reply_count": safe_get(t, "reply_count", 0),
        "quote_count": safe_get(t, "quote_count", 0),
        "possibly_sensitive": int(safe_get(t, "possibly_sensitive", False)),
        "is_quote_status": int(safe_get(t, "is_quote_status", False)),
        "lang_fr": int(safe_get(t, "lang", "") == "fr"),
    }

    # User features
    feats.update(extract_user_features(t.get("user", None)))

    # Quoted status features
    feats.update({
        "quoted_exists": int(quoted is not None),
    })

    # Quoted user features
    feats.update(extract_quoted_user_features(quoted))

    return feats


In [ ]:
def build_dataframe(raw):
    rows = []
    for tw in raw:
        text = extract_text(tw)
        feats = extract_tweet_features(tw)
        feats["text"] = text
        feats["challenge_id"] = tw["challenge_id"]
        rows.append(feats)
    return pd.DataFrame(rows)

df_train = build_dataframe(train_raw)
df_test  = build_dataframe(test_raw)

df_train["label"] = [tw["label"] for tw in train_raw]
df_train.head()


In [ ]:
num_features = [
    c for c in df_train.columns
    if c not in ["text", "label", "challenge_id"]
]

num_features = ['len_text',
  'has_extended',
  'n_hashtags',
  'n_urls',
  'n_mentions',
  'favorite_count',
  'retweet_count',
  'reply_count',
  'quote_count',
  #'possibly_sensitive',
  'is_quote_status',
  #'lang_fr',
  'user_followers',
  'user_friends',
  'user_statuses',
  'user_verified',
  'user_listed',
  'user_favourites',
  'user_geo_enabled',
  'quoted_exists',
  'q_user_followers',
  'q_user_friends',
  'q_user_statuses',
  'q_user_verified']

len(num_features), num_features


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_tr, df_val = train_test_split(
    df_train, 
    test_size=0.1, 
    stratify=df_train["label"], 
    random_state=42
    )

scaler = StandardScaler()

# Fit uniquement sur TRAIN
scaler.fit(df_train[num_features])

# Transform
df_train[num_features] = scaler.transform(df_train[num_features])
df_val[num_features]   = scaler.transform(df_val[num_features])
df_test[num_features]  = scaler.transform(df_test[num_features])

#df_train[num_features] = scaler.fit_transform(df_train[num_features])
#df_test[num_features]  = scaler.transform(df_test[num_features])

assert not df_train[num_features].isna().any().any(), "NaNs in TRAIN"
#assert not df_val[num_features].isna().any().any(),   "NaNs in VAL"
assert not df_test[num_features].isna().any().any(),  "NaNs in TEST"


In [ ]:
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False,        # BERTweet n’a PAS de tokenizer fast
    normalization=True     # important pour twitter (emoji, accents)
)

class TweetDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.features = df[num_features].values.astype(np.float32)
        self.labels = df["label"].tolist() if "label" in df else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        x = {k: v.squeeze(0) for k, v in enc.items()}
        feats = self.features[idx]

        if self.labels is None:
            return x, feats
        else:
            return x, feats, self.labels[idx]


In [ ]:


class FusionModel(nn.Module):
    def __init__(self, model_name, n_features, dropout=0.2):
        super().__init__()

        from transformers import RobertaModel
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden + n_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, x, feats):
        out = self.encoder(
            input_ids=x["input_ids"],
            attention_mask=x["attention_mask"]
        )
        cls = out.last_hidden_state[:, 0]   # [CLS]
        z = torch.cat([cls, feats], dim=1)
        logits = self.mlp(z)
        return logits


In [ ]:
from sklearn.model_selection import train_test_split

train_ds = TweetDataset(df_tr)
val_ds   = TweetDataset(df_val)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)


In [ ]:
model = FusionModel(MODEL_NAME, len(num_features)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=2e-5, 
    weight_decay=0.01
    )

num_epochs = 3
total_steps = num_epochs * len(train_dl)
warmup_steps = int(0.05 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

def run_epoch(dl, train, log_every=500):
    model.train() if train else model.eval()
    total_loss = 0
    correct = 0
    total = 0

    step = 0

    for batch in tqdm(dl):
        step += 1

        if train:
            optimizer.zero_grad()

        if train:
            (x, feats, y) = batch
        else:
            (x, feats, y) = batch

        x = {k: v.to(device) for k, v in x.items()}
        feats = feats.to(device)
        y = torch.tensor(y).float().unsqueeze(1).to(device)

        logits = model(x, feats)
        loss = criterion(logits, y)

        if train:
            loss.backward()
            optimizer.step()
            scheduler.step()

        # Accumulate logging stats
        batch_size = len(y)
        total_loss += loss.item() * batch_size
        preds = (logits.sigmoid() > 0.5).long()
        correct += (preds == y.long()).sum().item()
        total += batch_size

        # 🔥 Log every N steps
        if step % log_every == 0:
            current_loss = total_loss / total
            current_acc = correct / total
            if train:
                print(f"[TRAIN step {step}] loss={current_loss:.4f} acc={current_acc:.4f}")
            else:
                print(f"[VAL   step {step}] loss={current_loss:.4f} acc={current_acc:.4f}")

for epoch in range(num_epochs):
    print(f"\nEPOCH {epoch+1}")
    train_loss, train_acc = run_epoch(train_dl, True)
    val_loss, val_acc = run_epoch(val_dl, False)

    print(f"Train loss={train_loss:.4f} acc={train_acc:.4f}")
    print(f"Val   loss={val_loss:.4f} acc={val_acc:.4f}")


In [ ]:
test_ds = TweetDataset(df_test)
test_dl = DataLoader(test_ds, batch_size=32)

all_preds = []
all_ids = df_test["challenge_id"].tolist()

model.eval()
with torch.no_grad():
    for batch in tqdm(test_dl):
        x, feats = batch
        x = {k: v.to(device) for k, v in x.items()}
        feats = feats.to(device)

        logits = model(x, feats)
        probs = logits.sigmoid().cpu().numpy().flatten()
        all_preds.extend(probs.tolist())


In [ ]:
df_pred = pd.DataFrame({
    "challenge_id": all_ids,
    "prob": all_preds,
})

agg = df_pred.groupby("challenge_id")["prob"].mean().reset_index()
agg["Prediction"] = (agg["prob"] > 0.5).astype(int)
agg = agg[["challenge_id", "Prediction"]]
agg.columns = ["ID", "Prediction"]

agg.head()


In [ ]:
agg.to_csv("new_submission.csv", index=False)
agg.head()